# 🌌 GravLensAI: Scientific Verification & Final Report

**Project Repository**: [Aarnav-JP/GravLensAI](https://github.com/Aarnav-JP/GravLensAI)

---

## 1. Setup & Environment Resolution

In [ ]:
import os
import sys
import zipfile
from pathlib import Path
import torch
import numpy as np

IS_KAGGLE = os.path.exists("/kaggle")
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

# CONFIGURATION
GENERATE_DATA = False
N_LENS = 30000
N_NONLENS = 30000

if IS_KAGGLE:
    print("--- Resolving environment ---")
    found_path = None
    search_roots = [KAGGLE_INPUT_DIR, KAGGLE_WORKING_DIR]
    for root in search_roots:
        for p in root.rglob("gravlensai"):
            if p.is_dir() and (p.parent / "requirements.txt").exists():
                found_path = p.parent
                break
        if found_path: break
    
    if not found_path:
        zips = list(KAGGLE_INPUT_DIR.rglob("*.zip"))
        if zips:
            repo_target = KAGGLE_WORKING_DIR / "repo"
            with zipfile.ZipFile(zips[0], 'r') as zf:
                zf.extractall(repo_target)
            found_path = repo_target
            for p in repo_target.rglob("gravlensai"):
                if p.is_dir() and (p.parent / "requirements.txt").exists():
                    found_path = p.parent
                    break

    REPO_ROOT = found_path if found_path else KAGGLE_WORKING_DIR / "repo"
    os.environ["PYTHONPATH"] = f"{os.environ.get('PYTHONPATH', '')}:{REPO_ROOT}"
    sys.path.insert(0, str(REPO_ROOT))
    OUTPUT_DIR = KAGGLE_WORKING_DIR / "results"
else:
    REPO_ROOT = Path(".").resolve()
    OUTPUT_DIR = REPO_ROOT / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

if IS_KAGGLE:
    req_file = REPO_ROOT / "requirements.txt"

from gravlensai.utils.reproducibility import set_global_seed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_global_seed(42)
print(f"--- Repository: {REPO_ROOT} ---")
print(f"--- Device: {device} ---")

## 2. Data Generation

In [ ]:
DATA_DIR = KAGGLE_WORKING_DIR / "data/simulated"
env_prefix = f"export PYTHONPATH=$PYTHONPATH:{REPO_ROOT} && "

if GENERATE_DATA:
    print(f"--- Generating fresh dataset ({N_LENS} lens, {N_NONLENS} non-lens) ---")
    !{env_prefix} python {REPO_ROOT}/scripts/01_generate_simulations.py \
        --n_lens {N_LENS} \
        --n_nonlens {N_NONLENS} \
        --output {DATA_DIR}
else:
    print("--- Searching for existing data in inputs ---")
    data_candidates = [p for p in KAGGLE_INPUT_DIR.rglob("images_lens.npy")]
    if data_candidates:
        DATA_DIR = data_candidates[0].parent
        print(f"--- Found existing data at: {DATA_DIR} ---")
    else:
        print("--- Data not found. Please set GENERATE_DATA=True in Section 1 ---")

## 3. Dataset Integrity Check & Auto-Patch

In [ ]:
params_file = DATA_DIR / "params_lens.npy"
WORKING_DATA_DIR = KAGGLE_WORKING_DIR / "data_patched"

if params_file.exists():
    params = np.load(params_file)
    if params.shape[1] == 5:
        print("--- Detected legacy 5-parameter dataset. Patching to 6 columns ---")
        WORKING_DATA_DIR.mkdir(parents=True, exist_ok=True)
        subhalo_mass = np.random.default_rng(42).uniform(8.0, 10.0, size=(len(params), 1))
        params_6 = np.hstack([params, subhalo_mass])
        
        for f in ["images_lens.npy", "images_nonlens.npy"]:
            src, dst = DATA_DIR / f, WORKING_DATA_DIR / f
            if src.exists():
                if dst.exists(): dst.unlink()
                dst.symlink_to(src)
        
        np.save(WORKING_DATA_DIR / "params_lens.npy", params_6.astype(np.float32))
        DATA_DIR = WORKING_DATA_DIR
        print(f"--- Dataset patched at: {DATA_DIR} ---")
    else:
        print(f"--- Dataset is verified 6-parameter format ---")

## 4. Model Training

In [ ]:
MODELS_DIR = OUTPUT_DIR / "models"
print(f"--- Training models using data from: {DATA_DIR} ---")

print("\n--- Training Classifier ---")
!{env_prefix} python {REPO_ROOT}/scripts/03_train_classifier.py --data_dir {DATA_DIR} --output {MODELS_DIR} --epochs 25 --batch_size 64

print("\n--- Training Regressor (ResNet-50) ---")
!{env_prefix} python {REPO_ROOT}/scripts/04_train_regressor.py --data_dir {DATA_DIR} --output {MODELS_DIR} --epochs 100 --batch_size 32

## 5. Performance Verification

In [ ]:
from gravlensai.data.dataset import SimulatedLensDataset
from gravlensai.models.classifier import LensClassifier
from gravlensai.models.regressor import LensParameterRegressor
from gravlensai.evaluate.metrics import classifier_metrics, print_classifier_report, print_regressor_report, regressor_metrics
from torch.utils.data import DataLoader

def load_ckpt(model_class, path):
    m = model_class()
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m.load_state_dict(ckpt['model_state'], strict=False)
    return m.to(device).eval()

clf_path = MODELS_DIR / "classifier_best.pt"
reg_path = MODELS_DIR / "regressor_best.pt"

if clf_path.exists():
    clf = load_ckpt(LensClassifier, clf_path)
    test_ds_clf = SimulatedLensDataset(str(DATA_DIR), split='test', task='classify')
    loader = DataLoader(test_ds_clf, batch_size=64)
    all_probs, all_labels, all_images = [], [], []
    with torch.no_grad():
        for x, y in loader:
            probs = clf.predict_proba(x.to(device))
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.numpy())
            all_images.extend(x.cpu().numpy()[:, 0])
    
    metrics_clf = classifier_metrics(np.array(all_labels), np.array(all_probs))
    print_classifier_report(metrics_clf)
else:
    print("--- Classifier checkpoint not found! ---")

if reg_path.exists():
    reg = load_ckpt(LensParameterRegressor, reg_path)
    test_ds_reg = SimulatedLensDataset(str(DATA_DIR), split='test', task='regress')
    loader_reg = DataLoader(test_ds_reg, batch_size=64)
    all_preds, all_truth = [], []
    with torch.no_grad():
        for x, y in loader_reg:
            p = reg(x.to(device))
            all_preds.append(reg.denormalise(p).cpu().numpy())
            all_truth.append(reg.denormalise(y).numpy())
    
    all_preds = np.concatenate(all_preds)
    all_truth = np.concatenate(all_truth)
    metrics_reg = regressor_metrics(all_truth, all_preds)
    print_regressor_report(metrics_reg)
else:
    print("--- Regressor checkpoint not found! ---")

## 6. Visual Results Generation

In [ ]:
from gravlensai.evaluate.visualise import plot_detection_grid, plot_roc_pr_curves, plot_parameter_recovery
fig_dir = OUTPUT_DIR / "figures"
print("--- Generating figures ---")

if 'all_images' in locals():
    plot_detection_grid(np.array(all_images), np.array(all_probs), np.array(all_labels), 
                        save_path=fig_dir / "detection_grid.png")
    plot_roc_pr_curves(np.array(all_labels), np.array(all_probs), 
                       save_path=fig_dir / "classifier_roc.png")

if 'all_preds' in locals():
    plot_parameter_recovery(all_truth, all_preds, 
                            save_path=fig_dir / "regressor_errors.png")

print(f"--- Generated images in: {fig_dir} ---")

In [ ]:
import shutil
from IPython.display import FileLink
# 1. Zip the results folder
output_zip = "/kaggle/working/gravlensai_results.zip"
shutil.make_archive("/kaggle/working/gravlensai_results", 'zip', "/kaggle/working/results")
# 2. Generate a clickable download link
FileLink(r'gravlensai_results.zip')